In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from sklearn.metrics import roc_auc_score, log_loss, average_precision_score
from deepctr.feature_column import SparseFeat, DenseFeat, VarLenSparseFeat, get_feature_names
from deepctr.models import DeepFM
from numpy as np


In [5]:
import pandas as pd
from google.colab import drive
drive.mount('/content/drive')

train= pd.read_parquet('/content/drive/MyDrive/Colab Notebooks/CRT/train_part1.parquet' , engine= 'pyarrow')


Mounted at /content/drive


In [6]:
df = train.copy()

In [7]:
df = df.drop(['l_feat_20' , 'l_feat_23','l_feat_2','l_feat_24'], axis = 1 )
df = df.drop(['history_a_1' , 'history_a_2' , 'history_a_3'], axis= 1)
df = df[~df['inventory_id'].isin([92, 21])]

    max_len : 800
    topk : 1000
    min_count = 500

In [ ]:
df['seq'] = df['seq'].apply(lambda s: [int(t) for t in str(s).split(',') if t.strip()!=''])

df['seq'] = df['seq'].apply(lambda lst: [x+1 for x in lst if x >= 0])

df['seq_len_raw'] = df['seq'].apply(len)

In [ ]:
df[['seq','seq_len_raw']].head(5)

In [ ]:
# padding
from tensorflow.keras.preprocessing.sequence import pad_sequences
MAX_LEN = 150

seq_padding = (
    df['seq'],
    maxlen = MAX_LEN,
    dtype = 'int32',
    padding = 'post',
    truncating = 'pre',
    value = 0
)

df['seq_len'] = np.minimum(df['seq_len_raw'].values, MAX_LEN).astype('int32')


In [ ]:
sparse_feat = [
    SparseFeat('gender', vocabulary_size= 2 , embedding_dim= 4),
    SparseFeat('age_group' , vocabulary_size= 8 , embedding_dim= 8),
    SparseFeat('inventory_id' , vocabulary_size= 16, embedding_dim= 8 , embedding_name = 'lala'),
    SparseFeat('day_of_week', vocabulary_size=7 , embedding_dim= 8),
    SparseFeat('hour', vocabulary_size=24 , embedding_dim=16),
    SparseFeat('l_feat_14', vocabulary_size= 1131, embedding_dim= 16 ),
]

ordinal_sparse = [
    SparseFeat('l_feat_3' ,vocabulary_size= 3 , embedding_dim= 4),
    SparseFeat('l_feat_27',vocabulary_size= 5 , embedding_dim= 4),
    SparseFeat('feat_e_4' ,vocabulary_size= 4 , embedding_dim= 4),
    SparseFeat('feat_a_1' ,vocabulary_size= 5 , embedding_dim= 4),
    SparseFeat('feat_a_3' ,vocabulary_size= 6 , embedding_dim= 8),
    SparseFeat('feat_a_4' ,vocabulary_size= 6 , embedding_dim= 8),
    SparseFeat('feat_a_8' ,vocabulary_size= 7 , embedding_dim= 8),
    SparseFeat('feat_a_13',vocabulary_size= 5 , embedding_dim= 4),
    SparseFeat('feat_a_16',vocabulary_size= 7 , embedding_dim= 8),
    SparseFeat('feat_a_18',vocabulary_size= 7 , embedding_dim= 8)
]

# ordinal_scores = [
#     DenseFeat('l_feat_3_ordscore', 1),
#     DenseFeat('feat_a_1_ordscore', 1),

HASH_BUCKEY = 1_000_000
MAX_LEN = 150

varlen_seq  = VarLenSparseFeat(
    sparsefeat = SparseFeat('seq' ,
                            vocabulary_size= HASH_BUCKEY , #2715931
                            embedding_dim= 32 ,
                            use_hash=True ,
                            embedding_name = 'lala'),
    maxlen = MAX_LEN,
    combiner = 'mean',
    length_name= 'seq_len',
    weight_name = None,
    weight_norm = False
)


seq_col = 'seq'
label_col = 'clicked'
seq_len_col = 'seq_len'


nominal_names = [f.name for f in sparse_feat]
ordinal_names = [f.name for f in ordinal_sparse]

all_cols = df.columns.tolist()
exclude = set(nominal_names + ordinal_names + [label_col, seq_col, seq_len_col])

dense_feats = [DenseFeat(c, 1) for c in all_cols if c not in exclude]


In [ ]:
if dense_feats:
    mms = MinMaxScaler()
    df[dense_feats] = mms.fit_transform(df[dense_feats])


In [ ]:
# 모델 심층
dnn_feature_columns = varlen_seq
# 모델 선형
linear_feature_columns = sparse_feat + ordinal_sparse + dense_feats


feature_names = get_feature_names(linear_feature_columns + dnn_feature_columns)

# DIN: 고정길이 + 시퀀스(VarLenSparseFeat) 함께 사용
# dnn_feature_columns_din = linear_feature_columns + [varlen_seq]
# behavior_feature_list = ['inventory_id']  # query(현재 타깃)와 history 키 그룹 매칭
